# 双均线策略

策略介绍

双均线策略是一种常见的技术分析策略，基于两条不同时间周期的移动平均线（MA）来判断市场趋势并进行买卖操作。该策略的核心思想是通过较短周期和较长周期均线的交叉来捕捉价格趋势的变化。当短期均线向上穿越长期均线时，发出买入信号，预示价格可能进入上涨趋势；当短期均线向下穿越长期均线时，则发出卖出信号，提示价格可能开始下跌。双均线策略能够有效过滤掉市场的短期噪声，帮助投资者跟随中长期趋势进行交易。

在该策略中，我们选取了600519.SH（贵州茅台）作为标的股票。贵州茅台则是中国白酒行业的领军者，具有极高的品牌影响力和消费者忠诚度，长期来看股价也展现出较强的上升趋势。通过双均线策略，我们期望捕捉这只股票在价格趋势中的转折点，从而实现较为稳健的交易收益。

In [1]:
from bigmodule import M
import dai


In [8]:
# 交易引擎：初始化函数，只执行一次
def initialize(context):
    from bigtrader.finance.commission import PerOrder
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))  
    context.ins = context.instruments 

# 交易引擎：数据处理函数，每个周期执行一次
def handle_data(context, data):
    import pandas as pd
    from datetime import datetime, timedelta
    dt = data.current_dt.strftime('%Y-%m-%d')

    ma5 = data.history(context.ins[0], 'close', 5,'1d').mean() 
    ma80 = data.history(context.ins[0], 'close', 80,'1d').mean() 

    positions = context.get_account_positions()

    if (ma5 > ma80) and (context.ins[0]  not in positions):
        stock = context.ins[0]
        context.order_target_percent(stock, 0.5) 
        print(dt, '金叉买入 持半仓')

    
    elif (ma5 < ma80)  and (context.ins[0]  in positions):
        if positions[context.ins[0]].avail_qty > 0: 
            stock = context.ins[0]
            context.order_target_percent(stock, 0)
            print(dt, '死叉卖出 持空仓')
    


data = {
"start_date":'2015-01-01', 
'end_date':'2024-10-08', 
'market':'cn_stock',
'instruments':['600519.SH']

}

m3 = M.bigtrader.v30(
    data=data, 
    initialize=initialize,
    handle_data=handle_data,
    capital_base=500000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""open""",
    benchmark="""沪深300指数""",
    plot_charts=True
)

[2024-10-10 13:42:43] [info     ] bigtrader.v30 开始运行 ..
[2024-10-10 13:42:43] [info     ] 2015-01-01, 2024-10-08, cn_stock, equity, instruments=1
[2024-10-10 13:42:43] [info     ] bigtrader module V2.1.0
[2024-10-10 13:42:43] [info     ] bigtrader engine v1.10.10 2024-09-18


INFO:MAIN:======== bigtrader pid:2087 version 1.10.10 2024-09-18 ========
INFO:MAIN:bigtrader run_mode:BACKTEST, handle_bar_mode:0, frequency:1d, exchange_mode:BQ2
INFO:MAIN:> process add account:BACKTEST,0,bkt000
INFO:ACCT[bkt000]:AccountEngine: self_calc:1, validate_self_trading:0, validate_cash:0, validate_position:1, enable_auto_planed_order:1
INFO:MAIN:login_account(bkt000)
INFO:MAIN:> add_strategy setting:{'strategy_name': 'strategy', 'account_id': 'bkt000'}
INFO:MAIN:init all strategy account_id:...


2015-01-09 195.07199999999997 194.16333333333333 {} 金叉买入 持半仓
2015-01-13 189.264 192.03625 {'600519.SH': StockPosition(bkt000,600519.SH,direction:'1',current_qty:1300,avail_qty:1300,cost_price:190.057,last_price:184.88,margin:0.0,market_value:240344.0,open_date:20150112)} 死叉卖出 持空仓
2015-02-13 183.27800000000002 183.1938709677419 {} 金叉买入 持半仓
2015-02-17 182.948 183.16787878787878 {'600519.SH': StockPosition(bkt000,600519.SH,direction:'1',current_qty:1300,avail_qty:1300,cost_price:182.594762,last_price:182.22,margin:0.0,market_value:236886.0,open_date:20150216)} 死叉卖出 持空仓
2015-02-26 183.724 183.2897142857143 {} 金叉买入 持半仓
2015-07-20 241.06 244.480625 {'600519.SH': StockPosition(bkt000,600519.SH,direction:'1',current_qty:1455,avail_qty:1455,cost_price:167.0039,last_price:225.11,margin:0.0,market_value:327535.05000000005,open_date:20150227)} 死叉卖出 持空仓
2015-10-28 211.786 210.10437499999998 {} 金叉买入 持半仓
2016-01-08 208.37600000000003 210.00975000000003 {'600519.SH': StockPosition(bkt000,600519.SH,dir

INFO:MAIN:stop all strategy account_id:...


[2024-10-10 13:42:52] [info     ] backtest done, raw_perf_ds:dai.DataSource("_bcd61a6b2a614c78a89b100b093ed85c")


[2024-10-10 13:42:55] [info     ] bigtrader.v30 运行完成 [11.856s].
